# Итоговый проект по Python

Проект выполнен по данным `orders.xlsx` и `products.xlsx`.

В ноутбуке:
- загрузка и подготовка данных,
- расчёт всех требуемых метрик,
- построение графиков,
- итоговые выводы.

## 1. Импорт библиотек и базовые настройки

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Загрузка данных

In [ ]:
orders = pd.read_excel('orders.xlsx')
products = pd.read_excel('products.xlsx')

print('Размер orders:', orders.shape)
print('Размер products:', products.shape)

display(orders.head())
display(products.head())

## 3. Подготовка данных

Для всех заданий, кроме расчёта среднего чека, можно игнорировать товары из `orders`, которых нет в `products`.

In [ ]:
orders['accepted_at'] = pd.to_datetime(orders['accepted_at'])

orders['sales_amount'] = orders['price'] * orders['quantity']
orders['cost_amount'] = orders['cost_price'] * orders['quantity']
orders['promo_flag'] = orders['price'] != orders['regular_price']

df = orders.merge(products, on='product_id', how='inner')

df['sales_amount'] = df['price'] * df['quantity']
df['cost_amount'] = df['cost_price'] * df['quantity']
df['promo_flag'] = df['price'] != df['regular_price']

print('Размер после inner join:', df.shape)
display(df.head())

## 4. Самая ходовая товарная группа

Определим, по какой категории (`level1`) продано больше всего штук товара.

In [ ]:
category_sales_qty = (
    df.groupby('level1', as_index=False)
    .agg(sold_quantity=('quantity', 'sum'))
    .sort_values('sold_quantity', ascending=False)
    .reset_index(drop=True)
)

display(category_sales_qty)

top_category = category_sales_qty.loc[0, 'level1']
top_category_qty = category_sales_qty.loc[0, 'sold_quantity']

print(f'Самая ходовая товарная группа: {top_category} ({top_category_qty} шт.)')

### График: количество проданных штук по категориям

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(category_sales_qty['level1'], category_sales_qty['sold_quantity'])
plt.title('Количество проданных штук по товарным категориям')
plt.xlabel('Категория')
plt.ylabel('Количество проданных штук')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Распределение продаж по подкатегориям

Оценим распределение количества проданных позиций в каждой категории (`level1`) по подкатегориям (`level2`).

In [ ]:
subcategory_distribution = (
    df.groupby(['level1', 'level2'], as_index=False)
    .agg(sold_quantity=('quantity', 'sum'))
    .sort_values(['level1', 'sold_quantity'], ascending=[True, False])
    .reset_index(drop=True)
)

display(subcategory_distribution)

### Дополнительно: pivot-таблица по подкатегориям

In [ ]:
subcategory_pivot = (
    df.pivot_table(
        index='level2',
        columns='level1',
        values='quantity',
        aggfunc='sum',
        fill_value=0
    )
)

display(subcategory_pivot)

## 6. Средний чек на 13.01.2022

Для этого задания используем исходную таблицу `orders`, без обязательного джойна с `products`.

In [ ]:
target_date = pd.to_datetime('2022-01-13').date()

orders_target_day = orders[orders['accepted_at'].dt.date == target_date].copy()

check_totals = (
    orders_target_day.groupby('order_id', as_index=False)
    .agg(check_amount=('sales_amount', 'sum'))
)

avg_check = check_totals['check_amount'].mean()

display(check_totals.head())
print(f'Средний чек за {target_date.strftime("%d.%m.%Y")}: {avg_check:.2f} руб.')

## 7. Доля промо в категории «Сыры»

Считаем промо-продажи в штуках. Промо определяется как случай, когда `regular_price != price`.

In [ ]:
cheese_df = df[df['level1'] == 'Сыры'].copy()

promo_qty = cheese_df.loc[cheese_df['promo_flag'], 'quantity'].sum()
nonpromo_qty = cheese_df.loc[~cheese_df['promo_flag'], 'quantity'].sum()
total_cheese_qty = cheese_df['quantity'].sum()

promo_share = promo_qty / total_cheese_qty if total_cheese_qty != 0 else 0

promo_table = pd.DataFrame({
    'group': ['Промо', 'Не промо'],
    'quantity': [promo_qty, nonpromo_qty]
})
promo_table['share'] = promo_table['quantity'] / promo_table['quantity'].sum()

display(promo_table)
print(f'Доля промо в штуках: {promo_share:.2%}')

### График: доля промо в категории «Сыры»

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(
    promo_table['quantity'],
    labels=[f"{row['group']} ({row['share']:.1%})" for _, row in promo_table.iterrows()],
    autopct='%1.1f%%',
    startangle=90
)
plt.title('Доля промо-продаж в категории «Сыры» (в штуках)')
plt.tight_layout()
plt.show()

## 8. Маржа по категориям

Считаем маржу в рублях и в процентах по всем категориям `level1`.

In [ ]:
margin_by_category = (
    df.groupby('level1', as_index=False)
    .agg(
        revenue=('sales_amount', 'sum'),
        cost=('cost_amount', 'sum')
    )
)

margin_by_category['margin_rub'] = margin_by_category['revenue'] - margin_by_category['cost']
margin_by_category['margin_pct'] = np.where(
    margin_by_category['revenue'] != 0,
    margin_by_category['margin_rub'] / margin_by_category['revenue'] * 100,
    0
)

display(margin_by_category.sort_values('margin_rub', ascending=False))

### График: маржа по категориям, руб.

In [ ]:
margin_rub_plot = margin_by_category.sort_values('margin_rub', ascending=True).reset_index(drop=True)

plt.figure(figsize=(12, 8))
plt.barh(margin_rub_plot['level1'], margin_rub_plot['margin_rub'])
plt.title('Маржа по категориям, руб.')
plt.xlabel('Маржа, руб.')
plt.ylabel('Категория')
plt.tight_layout()
plt.show()

### График: маржа по категориям, %

In [ ]:
margin_pct_plot = margin_by_category.sort_values('margin_pct', ascending=True).reset_index(drop=True)

plt.figure(figsize=(12, 8))
plt.barh(margin_pct_plot['level1'], margin_pct_plot['margin_pct'])
plt.title('Маржа по категориям, %')
plt.xlabel('Маржа, %')
plt.ylabel('Категория')
plt.tight_layout()
plt.show()

## 9. ABC-анализ по подкатегориям

Проводим два анализа:
- по количеству продаж,
- по сумме продаж.

После этого формируем итоговую комбинированную группу, например `A C`.

In [ ]:
def abc_classification(dataframe, value_col, class_col_name):
    temp = dataframe.copy()
    temp = temp.sort_values(value_col, ascending=False).reset_index(drop=True)
    temp['share'] = temp[value_col] / temp[value_col].sum()
    temp['cum_share'] = temp['share'].cumsum()

    temp[class_col_name] = np.where(
        temp['cum_share'] <= 0.80, 'A',
        np.where(temp['cum_share'] <= 0.95, 'B', 'C')
    )

    if len(temp) > 0:
        temp.loc[0, class_col_name] = 'A'

    return temp

In [ ]:
abc_base = (
    df.groupby(['level1', 'level2'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('sales_amount', 'sum')
    )
)

display(abc_base.head())

In [ ]:
abc_qty = abc_classification(
    abc_base[['level1', 'level2', 'total_quantity']].copy(),
    'total_quantity',
    'abc_qty'
)

abc_sales = abc_classification(
    abc_base[['level1', 'level2', 'total_sales']].copy(),
    'total_sales',
    'abc_sales'
)

abc_result = (
    abc_base
    .merge(abc_qty[['level1', 'level2', 'abc_qty']], on=['level1', 'level2'], how='left')
    .merge(abc_sales[['level1', 'level2', 'abc_sales']], on=['level1', 'level2'], how='left')
)

abc_result['abc_final'] = abc_result['abc_qty'] + ' ' + abc_result['abc_sales']

abc_result = abc_result.sort_values(
    ['abc_qty', 'abc_sales', 'total_quantity', 'total_sales'],
    ascending=[True, True, False, False]
).reset_index(drop=True)

display(abc_result)

### Распределение итоговых ABC-групп

In [ ]:
abc_group_distribution = (
    abc_result['abc_final']
    .value_counts()
    .rename_axis('abc_final')
    .reset_index(name='count_subcategories')
)

display(abc_group_distribution)

## 10. Финальные выводы

In [ ]:
print('ИТОГОВЫЕ ВЫВОДЫ')
print(f'1. Самая ходовая категория: {top_category} ({top_category_qty} шт.)')
print(f'2. Средний чек 13.01.2022: {avg_check:.2f} руб.')
print(f'3. Доля промо в категории «Сыры»: {promo_share:.2%}')